# Technical note: Model validation and checking for cyclical dependencies

This note discusses cycle checking in Myokit's representation of a system of equations.

Cycles can occur in one of two ways:

1. An expression that references itself - no variables involved. This would mean that, instead of a nice expression tree, the code representation somehow contained a cycle. An example would be `a = Plus(1, 1)`, followed by `a._op1 = a`.
2. A variable with a right-hand side expression or initial value expression, that includes a reference to the same variable. An example would be `x = 1 - x` or `x = y; y = x`.

Below, we discuss both cases, followed by an audit of where these cycles **are** checked for and **should be** checked for, based on `Myokit 1.40.0`.

## Implementation detail: storing and caching references

To find all references to variables in an expression, we can traverse the expression tree and gather them.
However, Python is particularly slow at this kind of traversal, so Myokit caches references made by both _expressions_ and _variables_.

### References in expressions

Each `Expression` contains a `set` of all `LhsExpression` objects that appear in the tree rooted at that expression (with one important caveat, see below).

For example:, in the expression `x + (3 * y)` the `+` operator has a set `{x, y}` while the `*` operator has the set `{y}`

In [3]:
import myokit
times = myokit.parse_expression('3 * y')
plus = myokit.Plus(myokit.Name('x'), times)
print(times.references())
print(plus.references())

{<Name('y')>}
{<Name('y')>, <Name('x')>}


This set is created _at construction time_, and since expressions are immutable, it never changes.

Note that **children of LhsExpressions are not included**:

In [2]:
# Names include themselves in the set of references
ref = myokit.Name('a')
print(ref.references())

# And operators that are not LhsExpressions include all children
plus = myokit.Plus(ref, myokit.Name('b'))
print(plus.references())

# But LhsExpression types, like Derivative, do not list children
dot = myokit.Derivative(ref)
print(dot.references())

{<Name('a')>}
{<Name('b')>, <Name('a')>}
{<Derivative(<Name('a')>)>}


This behaviour matches the nature of dependencies: A dependency on `dx/dt` does not imply a dependency on `x`.

### References in variables

Variables maintain a two-way cache of references that depend on them.

As of Myokit 1.40.0, each variable has:
```
self._refs_by = set()   # Vars that refer to my LHS
self._srefs_by = set()  # Vars that refer to my state value
self._refs_to = set()   # Vars whos LHS I refer to
self._srefs_to = set()  # Vars whos state value I refer to
```

So in
```
dot(a) = 1
b = 2
c = b + dot(a)
d = b + a
```
the variable c's `_refs_to` will be `{b, a}`, while its `_srefs_to` will be empty.
The variable d's `_refs_to` will be `{b}`, while its `_srefs_to` will be `{c}`.

This behaviour matches the order of evaluation.
To calculate `c` we need to evaluate b and a's right-hand side.
To calculate `d` we need to evaluate b, but can use the current value of a.

## Case 1: Self-referencing expressions

Expressions should be structured as [trees](https://en.wikipedia.org/wiki/Tree_(graph_theory)).
This means that there should be no cycles.
More strongly, there should be exactly one path between any two nodes in the tree.

To guarantee this, Myokit expressions are defined as **immutable**, with their operands set at construction time.

As a result, direct self-reference, e.g. `x = 1 + x`, is not possible through the API.
```
x = Plus(x, y)  # redefines x, no self-reference!
```

But it can still be achieved by mutating an expression _outside of the API_:
```
x = Plus(Number(1), Number(1))
x._operands[1] = x
```

### Checking for Case 1 cycles

Versions up to v1.40.0 included an `Expression.validate()` method which checked for "case 1" cycles.
Beyond cycle checking, this method was used for 3 checks that could not be handled at construction time.
See [issue #1234](https://github.com/myokit/myokit/issues/1234).

This caused two problems:

1. A unit test for cycle checking could not be written without bypassing the API, and validation overhead included checking for this edge case which users sticking to the API could not encounter
2. To reduce overhead, expression validation results were cached. However, two of the three additional tests performed in validation were incorrectly cached - as their result could change after expression construction. In particular, they checked whether `Derivative` and `InitialValue` expressions referred to a state variable. Although expressions are immutable, the state-ness of a variable can be changed, and so this result should not be cached.

**After 1.40.0, Case 1 cycle checking was removed**. The  and the two problematic checks were moved to a private, non-cached method `_validate_for_variable`, to be called by the model during validation.

## Case 2: Self-reference via variables

This is much more common and achievable through the API.
Some simple examples are:

```
# Example 1
x = x

# Example 2
x = y
y = x
```

It is important to differentiate between state equations (differential equations) and state values.
The next two examples are not cycles:
```
# Example 3: not a cycle
dot(x) = x

# Example 4: not a cycle
dot(x) = y
y = x
```

But if instead we refer to the defining right-hand side expression of a state, we get cycles again:
```
dot(x) = y
y = dot(x)
```

### Scoping can be ignored in cycle checking

Myokit variables can only access variables within their scope.

```
[c1]
a = 2 * b
  b = 4
c = a
```
For example, in the above the variable `c` cannot access `b`.

**Scoping is resolved separately, before cycle checking. When cycle checking occurs, scope can be ignored.**

### Initial values

States also have an initial value, and since Myokit 1.34.0, this can be an expression involving constant variables.

The requirement that the variables are constant means an initial value can't reference its own state or derivative.

As there is no way to refer to an initial value, they can not be involved in cycles.

### Checking for Case 2 cycles

For historical reasons, Myokit checks for cycles in three different places.

#### `Model._validate_solvability()`

This is the method used by `Model.validate()`.
It works by creating a list of "used" variables (the states, bounds variables, and variables with labels), and then follows the dependencies listed _in var.rhs().references()_

#### `Model.solvable_order()`

This is the method used by most simulations when generating model code.

It returns the variables in "solvable order", but attempts to solve the system _component by component_.
If this can not be achieved, variables from "interdependent components" are grouped together.

To do this, it uses information from `Model.map_component_dependencies` to work out the component ordering.
Next, it uses information from `Model.map_shallow_dependencies` to work out variable ordering within each component.
Neither of these methods check for (or can encounter) cycles.

The `solvable_order` algorithm makes several passes over a list of unresolved variables.
If this list is unchanged after a pass, it means that there must be a cycle preventing the next variable from being added.
**However, it is assumed the model has been validated prior to running this method, so on failure no attempt to find the cycle is made and a `RuntimeError` is raised.**

#### `Model.component_cycles`

Myokit's multi-cell simulations require the model to be solvable component by component.
If this can not be achieved, the user should know which components depend on each other, and this is the problem solved by `component_cycles`.
It does _not_ check for cyclical dependencies between variables.




- Model.validate_solvability()
- Model.component_cycles()
- Model.map_deep_dependencies()
- Model.




### Methods that do not check.




# Questions

- How do we avoid unnecessary validation
  - Is it cached? Yes for expr. No for model status
  - When is it called?
- When are initial value expressions evaluated?
- Can we make cycles with `eval(subst=...)`
- Where do we check that initial values only reference (1) non-nested; and (2) constants.

## Implementation

The checks for cycles are run by

- Case 1: `Expression.validate()` --> But to be changed
- Case 2: `Model._validate_solvability()`
- Case 3: `Model._validate_solvability()`

lalala

- In `.validate()` methods, which are defined on `Expression`, `Variable`, `Component`, and `Model`
  - `Model.validate` calls `Component.validate`, which calls `Variable.validate`, which calls `.validate` on any child variables.
  - `Variable.validate` calls `Variable._rhs.validate` but does **not** call `init.validate()`

lololo

- In **XXXXXX**






## When is model.validate() called

1. When simulations are created
2. When HH or Markov models are created
3. Before exporting
4. After parsing (sets unused variable warnings)
5. Before creating python functions from Myokit equations: `Variable.pyfunc()`
6. When evaluating an RHS: `Model.show_evaluation_of`, `Variable.clamp`,
   - But **not** by `Variable.eval()`!
   - And **not** by `Expression.eval()`!
7. When evaluating derivatives: `Model.evaluate_derivatives` calls `solvable_order` 
8. When evaluating initial values: `Model.initial_values(as_floats=True)`, `Variable.initial_value(True)`

The last 2 can occur quite easily, e.g. when calling `float(var)`

## Validation required from within the expression system?

# TODO CHECK

- `eval()`
- `eval(subst)`
- `eval_unit()`
- `diff()`
- `diff()`-
- `depends_on()`
- `depends_on_state()`

# TODO CHECK FOR MORE

## Caching and invalidation

Full model validation (including solvability) is cached in `Model._is_valid` which is used as a tribool (`True`, `False`, `None`).

However, finding a model to be invalid _also_ raises a `myokit.IntegrityError`, so that the `False` value is rarely used.

The model validation status is reset by

- `Model.add_component` (and `add_component_allow_renaming`), `Model.remove_component`
- `Model.set_initial_values(x)` if `x` is a string or dict (handled by `map_to_state`)
- `Model._reset_validation()`, called by
  - `VarOwner.add_variable`, `add_variable_allow_renaming`
  - `VarOwner.move_variable`, `VarOwner.remove_variable`
  - `Variable.promote`, `Variable.demote`
  - `Variable.remove_child_variables()`
  - `Variable.set_binding` (bound variables are independent)
  - `Variable.set_rhs` and `Variable._set_initial_value`
  

Indirectly, it is set by methods such as `Variable.clamp` and others that call the above methods.



# TODO

- [x] ~Change expression validation of privates to asserts https://github.com/myokit/myokit/issues/1235~
- [x] `Variable.validate()` should validate initial value as well as RHS
- [ ] is_derivative missing () https://github.com/myokit/myokit/issues/1239
- [ ] Remove expression.validate() https://github.com/myokit/myokit/issues/1234
- [ ] Validate before eval() https://github.com/myokit/myokit/issues/1238
- [ ] Variable reffed only by init is marked unused https://github.com/myokit/myokit/issues/1163
- [ ] Bug in remove_child_variables() https://github.com/myokit/myokit/issues/1241
- [ ] Bug in Variable._delete https://github.com/myokit/myokit/issues/1242
- [ ] Reset validation after remove_child_variables
- [ ] Cache model validation
- [ ] Benchmark validation before and after changes

# WHERE IS INIT USED

## When exporting with "true"

```
./myokit/formats/ansic/template/euler.c:    print(tab + v(var) + ' = ' + myokit.float.str(var.initial_value(True)) + ';')

./myokit/formats/easyml/_exporter.py:                        + myokit.float.str(v.initial_value(True)) + eos)

./myokit/formats/opencl/template/cable.c:    print(tab*2 + '*s = ' + myokit.float.str(v.initial_value(True)) + '; s++; // ' + str(v.qname()))
```

**To-do**: Is it OK that the other exporters don't do this?

## Exporting without "true"

```
./myokit/formats/diffsl/_exporter.py:            rhs = e.ex(v.initial_value())
```

## Converting to HH/Markov

```
./myokit/lib/hh.py:            [v.initial_value(True) for v in self._states])
./myokit/lib/markov.py:            [v.initial_value(True) for v in self._states])
```

## Creating a CModel

```
./myokit/_sim/cmodel.h:    print(tab + v(var) + ' = ' + myokit.float.str(var.initial_value(True)) + ';')
```

## OTHER OTHER OTHER

```
./myokit/_model_api.py:                        [e.var() for e in var.initial_value().references()])
./myokit/_model_api.py:                state_map[var].initial_value().clone(subst=lhs_map))
./myokit/_model_api.py:            value = var.initial_value()
./myokit/_model_api.py:                value = self.initial_value(True)
./myokit/_model_api.py:            value = self.initial_value()
./myokit/_model_api.py:        return self.initial_value(as_float=True)
```

## Warnings

These are only raised for unused variables, in `Model._validate_solvability`.